[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bethavila17/dncp-riesgo-contrataciones/blob/main/notebooks/05_tiempo_casi_real_y_prototipo.ipynb)

# 5. Procesamiento casi en tiempo real y prototipo de consulta
**Proyecto III Big Data — Maestría en Ciencia de Datos (UCOM)** · Bethania Avila, Arturo Fernández, Gustavo Bogado

**Velocidad:** la API de la DNCP se actualiza cada hora. El sistema trae un micro-lote de procesos recién adjudicados, los integra al data lake, recalcula sus indicadores con el histórico y los puntúa con los modelos ya entrenados (sin reentrenar). Se mide la latencia de punta a punta.

In [1]:
# Configuración del entorno (Google Colab o local)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/bethavila17/dncp-riesgo-contrataciones.git"  # <- URL del repositorio en GitHub
USAR_DRIVE = False  # True: guarda el data lake en Google Drive y no repite descargas entre sesiones

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/dncp-riesgo-contrataciones")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-colab.txt")], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DNCP_DATA_DIR"] = "/content/drive/MyDrive/dncp_data_lake"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dncp_riesgo import graficos as gr
from dncp_riesgo.pipeline import asegurar_datos, preparar

cfg, lake = preparar()
gr.estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| años:", cfg["fuentes"]["ocp_registry"]["anios"])

Entorno: local | años: [2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [2]:
import time
asegurar_datos(lake, cfg, hasta="modelos")
from dncp_riesgo.ingesta import ingesta_incremental
from dncp_riesgo.modelos import ArtefactosRiesgo, ruta_modelos
from dncp_riesgo.procesamiento import construir_indicadores
artefactos = ArtefactosRiesgo.cargar(ruta_modelos(lake))
print("Detector:", artefactos.nombre_detector, "| Impugnación:", artefactos.nombre_supervisado, "| banderas del CRI:", len(artefactos.cri_banderas))

22:22:33 | dncp | Datos listos hasta la etapa 'modelos' (no se repite el procesamiento).


Detector: Isolation Forest global | Impugnación: LightGBM | banderas del CRI: 5


In [3]:
t0 = time.perf_counter()
info = ingesta_incremental(lake, cfg, dias=3, max_procesos=15)          # 1. captura (API, micro-lote)
t1 = time.perf_counter()
ind = construir_indicadores(lake, cfg)                                  # 2. indicadores con el histórico
t2 = time.perf_counter()
nuevos = ind[ind.ocid.isin(info["ocids"]) & (ind.adjudicado == 1)]
alertas = nuevos[["ocid", "comprador_nombre", "proveedor_nombre", "monto_adjudicado"]].join(artefactos.puntuar(nuevos))  # 3. puntaje
t3 = time.perf_counter()
pd.Series({"captura por API (rate limit 15/min)": t1 - t0, "indicadores (todo el histórico)": t2 - t1,
           "puntuación + explicaciones": t3 - t2, "TOTAL": t3 - t0}, name="segundos").round(1).to_frame()

22:23:10 | dncp | API DNCP: 317 procesos encontrados, se descargan 15


22:23:39 | dncp | API DNCP /ocds/record/ocds-03ad3f-446762-1 (intento 1/4): ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))


22:23:58 | dncp | API DNCP /ocds/record/ocds-03ad3f-446762-1 (intento 2/4): ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))


22:24:31 | dncp | API DNCP /ocds/record/ocds-03ad3f-444653-1 (intento 1/4): ('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))


22:25:20 | dncp |   10/15 registros


22:26:57 | dncp | Indicadores: 58621 procesos, 88 variables (64.2 s)


,segundos
captura por API (rate limit 15/min),194.3
indicadores (todo el histórico),67.6
puntuación + explicaciones,27.5
TOTAL,289.4


In [4]:
alertas.sort_values("prioridad", ascending=False)[["comprador_nombre", "proveedor_nombre", "monto_adjudicado", "prioridad", "nivel", "pct_impugnacion", "banderas_activas", "factores_anomalia"]]

,comprador_nombre,proveedor_nombre,monto_adjudicado,prioridad,nivel,pct_impugnacion,banderas_activas,factores_anomalia
37229,Petróleos Paraguayos (PETROPAR),Trafigura Pte Ltd,2.383572e+11,97.6,Alto,80.8,"Un solo oferente en un procedimiento competitivo; Precio unitario adjudicado > 1,5 vec...",Comprador nivel: Empresas Públicas; Precio unitario / precio de mercado; Desvío estima...
30498,Petróleos Paraguayos (PETROPAR),Trafigura Pte Ltd,1.467157e+11,95.1,Alto,81.1,El proveedor concentra >= 25 % de lo adjudicado por la entidad en los 12 meses previos,Comprador nivel: Empresas Públicas; Subasta electrónica; % de ampliación del contrato
15248,Ministerio de Defensa Nacional (MDN),ELTA SYSTEMS LTD,9.451728e+09,91.8,Alto,59.2,"Contratación por vía de la excepción; Precio unitario adjudicado > 1,5 veces la median...",Proveedor nuevo; Metodo grupo: Excepción; Cantidad de lotes
13357,Instituto Paraguayo de Tecnologia Agraria (IPTA),Fuentes Coiana Gonzalo Javier (Espacio Pliegues),1.087854e+09,88.3,Alto,40.9,Un solo oferente en un procedimiento competitivo; Monto adjudicado supera al estimado;...,Proveedor nuevo; Publicado en nov./dic.; Comprador nivel: Entes Autónomos y Autárquicos
30630,Petróleos Paraguayos (PETROPAR),VITOL ENERGIA AMERICAS S.A,1.206454e+11,76.9,Medio,91.2,"Precio unitario adjudicado > 1,5 veces la mediana de mercado del ítem",Comprador nivel: Empresas Públicas; Subasta electrónica; Ofertas previas del proveedor
28136,Agencia Financiera de Desarrollo (AFD),GESTORINCSA S.A.,5.998330e+08,75.9,Medio,78.2,Un solo oferente en un procedimiento competitivo,Comprador nivel: Entidades Financieras Oficiales; Ofertas previas del proveedor; Plazo...
30542,Petróleos Paraguayos (PETROPAR),"PAN AMERICAN ENERGY, S.L., SUCURSAL ARGENTINA",1.784364e+10,75.5,Medio,89.0,,Comprador nivel: Empresas Públicas; Subasta electrónica; Ofertas previas del proveedor
30439,Petróleos Paraguayos (PETROPAR),"PAN AMERICAN ENERGY, S.L., SUCURSAL ARGENTINA",1.454849e+10,75.2,Medio,87.3,,Comprador nivel: Empresas Públicas; Subasta electrónica; Días de adjudicación a contrato
12025,Corte Suprema de Justicia (CSJ),COASEGURO ASEGURADORA YACYRETA S.A./PATRIA S.A. DE SEGUROS Y REASEGUROS/ASEGURADORA TA...,1.341330e+10,72.2,Medio,92.0,Proveedor sin historial gana un monto alto (percentil 90 de su categoría),Comprador nivel: Poder Judicial; Proveedor nuevo; Enmiendas del contrato
5422,Dirección Nacional de Aeronáutica Civil (DINAC),COASEGURO ASEGURADORA YACYRETA S.A./ASEGURADORA DEL ESTE S.A./PATRIA S.A. DE SEGUROS.-,7.115243e+09,71.4,Medio,25.6,Proveedor sin historial gana un monto alto (percentil 90 de su categoría),Comprador nivel: Empresas Públicas; Proveedor nuevo; Sin etapa de planificación


## Prototipo de consulta (Streamlit)
Ranking, detalle de cada proceso con sus factores, tendencias, redes y métricas.

- **Local:** `streamlit run app/streamlit_app.py`
- **Colab:** cambiar `LANZAR_APP = True`; se publica con un túnel temporal de Cloudflare (la URL aparece en la salida).

In [5]:
LANZAR_APP = False
if LANZAR_APP and EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)
    subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared", shell=True)
    subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", "8501", "--server.headless", "true"])
    tunel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], stderr=subprocess.PIPE, text=True)
    for linea in tunel.stderr:
        if "trycloudflare.com" in linea:
            print("Abrir:", linea.split()[-2] if "|" in linea else linea.strip()); break